# 21: Attention Visualization - See What Models See

## Making the Invisible Visible

One of attention's superpowers: **interpretability**! We can visualize **what the model focuses on**.

This notebook explores different ways to visualize attention weights:
- Heatmaps
- Line plots
- Interactive visualizations

### The Web Dev Analogy

Attention visualization is like **DevTools Network tab**:
- See which resources (words) are accessed
- Understand dependencies between components
- Debug unexpected behavior
- Optimize performance by seeing bottlenecks

## What You'll Learn
- [ ] Interpret attention weight matrices as heatmaps
- [ ] Identify meaningful attention patterns (syntactic, semantic, positional)
- [ ] Use attention weights to explain model predictions

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 20**: Attention computes weights over input tokens | Now we visualize those weights to understand *what* the model focuses on |
| **Lesson 15**: Embedding visualization | Same idea — making the model's internal representations interpretable |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

print("Ready to visualize attention! 🎨")

## 1. Simple Attention Heatmap

In [ ]:
# Simulate attention weights for a sentence
sentence = "The movie was not very good"
words = sentence.split()
n_words = len(words)

# Simulate attention pattern (each word attending to others)
# Row i = attention weights when word i is the query
np.random.seed(42)
attention_weights = np.random.rand(n_words, n_words)

# Normalize rows to sum to 1
attention_weights = attention_weights / attention_weights.sum(axis=1, keepdims=True)

# Make it more interesting: "not" and "very" should attend to "good"
attention_weights[3, 5] = 0.6  # "not" → "good"
attention_weights[4, 5] = 0.5  # "very" → "good"
attention_weights = attention_weights / attention_weights.sum(axis=1, keepdims=True)

print(f"Sentence: {sentence}")
print(f"Attention matrix shape: {attention_weights.shape}")
print(f"\nEach row sums to 1: {attention_weights.sum(axis=1).round(3)}")

In [ ]:
# Visualize as heatmap
plt.figure(figsize=(10, 8))

sns.heatmap(
    attention_weights,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    xticklabels=words,
    yticklabels=words,
    cbar_kws={'label': 'Attention Weight'},
    square=True
)

plt.xlabel('Attending TO (Keys)', fontsize=12, fontweight='bold')
plt.ylabel('Attending FROM (Queries)', fontsize=12, fontweight='bold')
plt.title('Attention Weights Heatmap', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n💡 Reading the heatmap:")
print("   - Row i: When word i is the query")
print("   - Column j: How much attention to word j")
print("   - Bright = high attention, Dark = low attention")

`★ Insight ─────────────────────────────────────`

**What attention patterns reveal:**
1. **Syntax**: Which words depend on others grammatically
2. **Semantics**: Which words are semantically related
3. **Long-range**: Dependencies across the sequence
4. **Debugging**: Why model made a prediction

This interpretability is one of attention's killer features!

`─────────────────────────────────────────────────`

## 2. Single Query Attention

In [ ]:
# Focus on one word's attention
query_word_idx = 3  # "not"
query_word = words[query_word_idx]
query_attention = attention_weights[query_word_idx]

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(words, query_attention, color='steelblue', edgecolor='black', linewidth=1.5)

# Highlight the highest attention
max_idx = query_attention.argmax()
bars[max_idx].set_color('coral')
bars[max_idx].set_edgecolor('darkred')

ax.set_ylabel('Attention Weight', fontsize=12, fontweight='bold')
ax.set_title(f'Attention from "{query_word}"', fontsize=14, fontweight='bold')
ax.set_ylim([0, max(query_attention) * 1.2])
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, (word, weight) in enumerate(zip(words, query_attention)):
    ax.text(i, weight + 0.01, f'{weight:.2f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print(f"\n'{query_word}' pays most attention to: '{words[max_idx]}'")
print(f"This makes sense! Negation word looks at what it's negating.")

## 3. Sequence-to-Sequence Attention

In [ ]:
# Translation example: English to French
source_sentence = "I love machine learning"
target_sentence = "J'adore apprentissage automatique"

source_words = source_sentence.split()
target_words = target_sentence.split()

# Simulate cross-attention (target attending to source)
# Shape: (target_len, source_len)
cross_attention = np.array([
    [0.8, 0.1, 0.05, 0.05],  # "J'adore" → "I"
    [0.1, 0.8, 0.05, 0.05],  # "adore" → "love"
    [0.05, 0.1, 0.7, 0.15],  # "apprentissage" → "machine"
    [0.05, 0.05, 0.2, 0.7],  # "automatique" → "learning"
])

print(f"Source: {source_sentence}")
print(f"Target: {target_sentence}")
print(f"\nCross-attention shape: {cross_attention.shape}")
print(f"  (target_len, source_len)")

In [ ]:
# Visualize cross-attention
plt.figure(figsize=(10, 6))

sns.heatmap(
    cross_attention,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=source_words,
    yticklabels=target_words,
    cbar_kws={'label': 'Attention Weight'},
    linewidths=0.5,
    linecolor='gray'
)

plt.xlabel('Source (English)', fontsize=12, fontweight='bold')
plt.ylabel('Target (French)', fontsize=12, fontweight='bold')
plt.title('Cross-Attention in Translation', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n💡 Cross-attention shows alignment:")
print("   - Each target word attends to relevant source words")
print("   - Diagonal pattern = similar word order")
print("   - Non-diagonal = different word order between languages")

## 4. Multi-Head Attention Visualization

In [ ]:
# Simulate multiple attention heads
sentence = "The cat sat on the mat"
words = sentence.split()
n_words = len(words)
n_heads = 4

# Generate different attention patterns for each head
np.random.seed(42)
attention_heads = []

for head in range(n_heads):
    # Each head learns different patterns
    attn = np.random.rand(n_words, n_words)
    
    # Make each head specialize
    if head == 0:  # Syntax head: attend to nearby words
        for i in range(n_words):
            attn[i, max(0, i-1):min(n_words, i+2)] += 2
    elif head == 1:  # Subject-verb head
        attn[1, 2] += 3  # "cat" → "sat"
        attn[2, 1] += 3  # "sat" → "cat"
    elif head == 2:  # Preposition head
        attn[3, 5] += 3  # "on" → "mat"
        attn[5, 3] += 3  # "mat" → "on"
    else:  # Long-range head
        attn[0, -1] += 2
        attn[-1, 0] += 2
    
    # Normalize
    attn = attn / attn.sum(axis=1, keepdims=True)
    attention_heads.append(attn)

# Visualize all heads
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

head_names = ['Syntax (local)', 'Subject-Verb', 'Preposition', 'Long-range']

for idx, (attn, name) in enumerate(zip(attention_heads, head_names)):
    sns.heatmap(
        attn,
        annot=True,
        fmt='.2f',
        cmap='viridis',
        xticklabels=words,
        yticklabels=words,
        cbar_kws={'label': 'Weight'},
        ax=axes[idx],
        square=True,
        linewidths=0.5
    )
    axes[idx].set_title(f'Head {idx+1}: {name}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Different heads learn different patterns!")
print("   Each head specializes in different aspects of language.")

`★ Insight ─────────────────────────────────────`

**Why multi-head attention:**
1. **Different heads = different perspectives**
2. One head might focus on syntax, another on semantics
3. Allows model to attend to multiple things simultaneously
4. Increases representational capacity

Like having multiple team members each focused on different aspects!

`─────────────────────────────────────────────────`

## 5. Attention Flow Over Layers

In [ ]:
# Visualize how attention evolves through layers
sentence = "The quick brown fox"
words = sentence.split()
n_words = len(words)
n_layers = 3

# Simulate attention at different layers
np.random.seed(42)
layer_attentions = []

for layer in range(n_layers):
    attn = np.eye(n_words) * 0.3  # Start with self-attention
    
    # Layer 0: Local patterns
    if layer == 0:
        for i in range(n_words-1):
            attn[i, i+1] = 0.4
            attn[i+1, i] = 0.3
    
    # Layer 1: Medium range
    elif layer == 1:
        attn[0, 2] = 0.4  # "The" → "brown"
        attn[1, 3] = 0.4  # "quick" → "fox"
    
    # Layer 2: Long range, more abstract
    else:
        attn[0, 3] = 0.5  # "The" → "fox"
        attn[3, 0] = 0.4  # "fox" → "The"
    
    # Normalize
    attn = attn / attn.sum(axis=1, keepdims=True)
    layer_attentions.append(attn)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, attn in enumerate(layer_attentions):
    sns.heatmap(
        attn,
        annot=True,
        fmt='.2f',
        cmap='RdYlGn',
        xticklabels=words,
        yticklabels=words,
        cbar=True,
        ax=axes[idx],
        square=True,
        vmin=0,
        vmax=0.5
    )
    axes[idx].set_title(f'Layer {idx+1}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Attention patterns evolve through layers:")
print("   Layer 1: Local, syntax-level patterns")
print("   Layer 2: Medium-range dependencies")
print("   Layer 3: Long-range, semantic patterns")
print("\n   Lower layers = simple patterns")
print("   Higher layers = complex, abstract relationships")

## 6. Attention Statistics

In [ ]:
# Analyze attention patterns
attn = attention_heads[0]  # Use first head from earlier

# Compute statistics
max_attention = attn.max(axis=1)
entropy = -np.sum(attn * np.log(attn + 1e-9), axis=1)
self_attention = np.diag(attn)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Max attention per word
axes[0].bar(words, max_attention, color='steelblue')
axes[0].set_title('Max Attention Weight per Word', fontweight='bold')
axes[0].set_ylabel('Max Weight')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')

# Attention entropy (how spread out is attention?)
axes[1].bar(words, entropy, color='coral')
axes[1].set_title('Attention Entropy (Spread)', fontweight='bold')
axes[1].set_ylabel('Entropy')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

# Self-attention
axes[2].bar(words, self_attention, color='green')
axes[2].set_title('Self-Attention Weight', fontweight='bold')
axes[2].set_ylabel('Weight')
axes[2].tick_params(axis='x', rotation=45)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  High max attention: Word focuses heavily on one specific word")
print("  High entropy: Attention is spread across many words")
print("  High self-attention: Word mostly attends to itself")

## 7. Practical Tips for Attention Visualization

In [ ]:
print("Best Practices for Attention Visualization:")
print("=" * 70)

print("\n1. Choose the Right Visualization:")
print("   - Heatmap: Full attention matrix")
print("   - Bar chart: Single query's attention")
print("   - Line plot: Attention over time")
print("   - Network graph: For very sparse attention")

print("\n2. Normalize Properly:")
print("   - Attention weights should sum to 1 (softmax)")
print("   - Use consistent color scale across visualizations")

print("\n3. What to Look For:")
print("   - Diagonal patterns: Sequential dependencies")
print("   - Vertical/horizontal lines: Important words")
print("   - Blocks: Phrase-level attention")
print("   - Uniform: Model is uncertain")

print("\n4. Common Patterns:")
print("   - Stop words (the, a): Often low attention")
print("   - Content words (nouns, verbs): Higher attention")
print("   - Function words attending to content words")

print("\n5. Tools:")
print("   - BertViz: Interactive attention visualization")
print("   - exBERT: Explore BERT attention patterns")
print("   - Transformers Interpret: Model interpretation library")

## 📝 Check Your Understanding

1. How do you read an attention heatmap?
2. What does high entropy in attention weights indicate?
3. Why do different attention heads show different patterns?
4. What's the difference between self-attention and cross-attention visualization?
5. How can attention visualization help debug models?

In [ ]:
# --- Exercise 1: Read an Attention Matrix ---
# Given an attention matrix, find which input token gets the most attention
# for the first output position (row 0).

attention_matrix = np.array([
    [0.1, 0.05, 0.8, 0.05],   # Output token 0 attends to...
    [0.6, 0.2, 0.1, 0.1],     # Output token 1 attends to...
    [0.05, 0.05, 0.05, 0.85],  # Output token 2 attends to...
])

# YOUR CODE HERE:
most_attended_by_0 = None  # Index of the most-attended input for output position 0

# --- Check ---
assert most_attended_by_0 is not None, "Use np.argmax on the first row!"
assert most_attended_by_0 == 2, f"Output 0 gives 0.8 weight to input 2. Got index {most_attended_by_0}"
print("Exercise 1 passed! ✓")

# --- Quick Check: Interpreting Attention ---
# If attention weights for a token are [0.01, 0.01, 0.96, 0.02], what can we say?
# a) The model is confused and uncertain
# b) The model is strongly focusing on the 3rd input token
# c) The model is attending to all tokens equally
# d) The attention mechanism is broken

your_answer = None  # Put 'a', 'b', 'c', or 'd'

# --- Check ---
assert your_answer is not None, "Pick an answer!"
assert your_answer == 'b', "0.96 on one token = almost all attention on that one token!"
print("Exercise 2 passed! ✓")

print("\n🎉 All exercises passed!")

## 🎯 Summary

**Attention visualization techniques**:
- **Heatmaps**: Full attention matrix
- **Bar charts**: Single query focus
- **Multi-head**: Different attention perspectives
- **Layer-wise**: Evolution of attention patterns

**Key insights from visualization**:
1. Which words model focuses on
2. How attention patterns differ by head
3. Evolution of patterns through layers
4. Alignment in sequence-to-sequence tasks

**Why it matters**:
- **Interpretability**: Understand model decisions
- **Debugging**: Identify issues in model behavior
- **Trust**: Build confidence in model outputs
- **Research**: Discover what models learn

**Visualization reveals**:
- Syntactic dependencies
- Semantic relationships
- Long-range connections
- Model specialization

**Next up**: Sequence-to-sequence with attention! →